# 01 · Additive LightGBM at exact-value resolution

**Role in the stack:** our own core model. Single-model CV **0.94540**, public LB 0.94567. Final stack weight 0.058, but it is the
model every other member is measured against, and the one that taught us what the data is.

## The idea
The competition data is a 668k-row synthetic copy of a 10k-row original that was itself generated by a logistic formula
(Chris Deotte recovered it: `1.2*income/1e5 + 0.6*concern + 2*subsidy - 1*[anxiety=Medium] - 3*[anxiety=High] + noise > 5.5`).
Two facts follow, and we proved both on our folds before building anything else:

1. **The generator is additive in the logit.** A residual audit of the additive model over all 78 feature pairs found nothing,
   while a synthetic positive control (an injected pair effect) was detected 25x stronger. So trees that split on feature
   interactions are spending capacity on noise.
2. **The signal lives in exact values.** `Annual_Income_USD` has 13,214 distinct values and the generator wrote label signal into
   them (every income value is in effect an ID of an original row). LightGBM's default 255 histogram bins merge ~50 income
   values per bin and throw that signal away.

Two parameters implement both facts:
- `max_bin=13214` gives every distinct income its own bin, so a split can isolate one exact value.
- `interaction_constraints=[[0],[1],...,[12]]` lets each tree split on ONE feature only, so the ensemble is a sum of
  per-feature functions: an additive model, exactly the generator's form.

Together: **0.94163 (baseline LGBM) -> 0.94541**, the single biggest step of the competition. Everything else we tried on this
model afterwards (semantic features, pairwise interactions, target/frequency encoding, capacity and learning-rate sweeps, seed
bagging, dropping weak columns) measured at or below the noise floor.

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
from cv import load, run_cv, paired, SEED, NFOLDS, OOF, TEST
X, y, Xte, te_id = load()          # data/train.csv, data/test.csv  (kaggle competitions download -c playground-series-s6e9)
print(X.shape, Xte.shape, 'positives %.4f' % y.mean())

In [ ]:
import lightgbm as lgb
from feats import additive

# Slow learner, fixed tree count. Chosen from fold-0 early stopping at lr .02 (plateau 8092 trees), scaled to lr .006.
# No early stopping on the validation fold -> the OOF is honest and stackable.
LR, N = 0.006, 27000
P = dict(n_estimators=N, learning_rate=LR, num_leaves=31,
         max_bin=13214,                                   # one bin per distinct income value
         interaction_constraints=additive(X.shape[1]),    # one feature per tree -> additive in the logit
         verbose=-1, random_state=SEED)
print(P)

In [ ]:
# ~50 min on CPU. run_cv = the fixed StratifiedKFold(5, shuffle=True, random_state=42) every member of the stack uses.
oof, _ = run_cv(lambda: lgb.LGBMClassifier(**P), X, y, 'additive lgbm')
np.save(OOF / 'oof_final.npy', oof)

# Full-data refit for the test prediction: 10% more rows -> 10% more trees.
m = lgb.LGBMClassifier(**dict(P, n_estimators=int(N * 1.1))).fit(X, y)
np.save(TEST / 'test_final.npy', m.predict_proba(Xte)[:, 1])

## What we learned from this model
- A gain measured on a fast proxy config is a hypothesis, not a result. The core-6 feature subset was +0.0003 at lr .02 and
  **-0.0002** at production lr .006 (the slow learner shrinks the 7 weak features and still harvests their small real signal).
- Seed bagging the additive model gains exactly 0: with one feature per tree there is no randomness that matters.
- The per-exact-value signal has a median of 8 rows per income value. A tree with `min_data_in_leaf=20` and range splits pools
  each value toward its sorted neighbours. We later built a direct per-value logistic model (`experiments/direct_additive.py`
  in the workspace) and learned that shrinking those effects toward zero loses 0.004; shrinking toward neighbours recovers most
  of it. That is why `max_bin=13214` was the whole competition for a single model.